In [0]:
#!pip install pandas openpyxl
#!pip install pyxlsb #

In [0]:
from os import listdir
from datetime import datetime,date
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType, IntegerType, StringType,FloatType,DateType,TimestampType, StructField, StructType
from datetime import datetime,date,timedelta
from dateutil.relativedelta import relativedelta
from pyspark.sql.window import Window
import pandas as pd
from functools import reduce
from pyspark.sql import DataFrame
import pytz
import csv
import numpy as np

Se usara el esquema "mx_unilever_pbl".

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mx_unilever_pbl;
USE SCHEMA mx_unilever_pbl;

In [0]:
def rename_columns(df,lista):
  size_list=len(lista)
  if size_list>0:
    head,tail = (lambda lista : (lista[0],lista[1:]))(lista)
    if head:
      df_clean = df.withColumnRenamed(head,head.replace("sum(","").replace(")","").replace("+","_").replace(" ","_").replace("-","_").replace(".",""))
      return rename_columns(df_clean,tail)
  else:
    return df
  
def clean_numbers(df, columns):
    for column in columns:
        df = df.withColumn(
            column,
            F.regexp_replace(F.col(column), r"\.", "")  
        ).withColumn(
            column,
            F.regexp_replace(F.col(column), r",", ".") 
        ).withColumn(
            column,
            F.col(column).cast(DoubleType())
        )
    return df


# Leer datos

### a) Cargar MARM


In [0]:
marm = pd.read_csv("/dbfs/mnt/unilever/PBL/MARM.txt",skiprows=1,  encoding='latin1', sep="\t", quoting = csv.QUOTE_NONE)
marm.columns=marm.columns.str.strip()
marm = marm.replace({np.nan: None})

/root/.ipykernel/62684/command-4533557775654279-1154924906:1: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  marm = pd.read_csv("/dbfs/mnt/unilever/PBL/MARM.txt",skiprows=1,  encoding='latin1', sep="\t", quoting = csv.QUOTE_NONE)


In [0]:
marm = spark.createDataFrame(marm[["Material", "UMA", "Numerad.","Denom.", "Volumen", "UV", "Bruto", "Un"]])

marm_2 = rename_columns(marm,marm.columns)
marm_2 = clean_numbers(marm_2, ["Numerad", "Denom", "Volumen", "Bruto"])

/databricks/spark/python/pyspark/sql/pandas/conversion.py:473: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  Could not convert '40079400027182' with type str: tried to convert to int64
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)



## Limpieza de datos

### a) Cajas

In [0]:
cat_df = marm_2.filter(F.col("UMA") == "CS") \
    .select("Material",
            F.lit(0).alias("CajasxTSP"),
            F.lit(0).alias("CajasxLayer"),
            F.lit(0).alias("CajasxPallet"),
            F.lit(0).alias("LayersxPallet"),
            F.col("Volumen").alias("VolCS"),
            F.lit(0).alias("VolLay"),
            F.lit(0).alias("VolPal"),
            F.col("UV").alias("VolUnid"),
            F.col("Bruto").alias("PesoCS"),
            F.lit(0).alias("PesoLay"),
            F.lit(0).alias("PesoPal"),
            F.col("Un"))

### b) Layers

In [0]:
lay_df = marm_2.filter(F.col("UMA") == "LAY").select("Material", "Numerad", "Volumen", "Bruto")

cat_df = cat_df.alias("A").join(lay_df.alias("B"), "Material", "left") \
    .withColumn("CajasxLayer", F.coalesce(F.col("B.Numerad"), F.col("CajasxLayer"))) \
    .withColumn("VolLay", F.coalesce(F.col("B.Volumen"), F.col("VolLay"))) \
    .withColumn("PesoLay", F.coalesce(F.col("B.Bruto"), F.col("PesoLay")))

### c) Pallets

In [0]:
pal_df = marm_2.filter(F.col("UMA") == "PAL").select("Material", "Numerad", "Volumen", "Bruto")

cat_df = cat_df.alias("A").join(pal_df.alias("B"), "Material", "left") \
    .withColumn("CajasxPallet", F.coalesce(F.col("B.Numerad"), F.col("CajasxPallet"))) \
    .withColumn("VolPal", F.coalesce(F.col("B.Volumen"), F.col("VolPal"))) \
    .withColumn("PesoPal", F.coalesce(F.col("B.Bruto"), F.col("PesoPal")))

### d) TSP 

In [0]:
tsp_df = marm_2.filter(F.col("UMA") == "TSP").select("Material", "Numerad", "Denom", "Volumen", "Bruto")

cat_df = cat_df.alias("A").join(tsp_df.alias("B"), "Material", "left") \
    .withColumn("CajasxTSP", F.coalesce(F.try_divide("B.Numerad", "B.Denom"), F.col("CajasxTSP"))) 

### e) Layers por Pallet

In [0]:
cat_df = cat_df.withColumn("LayersxPallet", 
                           F.when(F.col("CajasxLayer") > 0, F.col("CajasxPallet") / F.col("CajasxLayer"))
                           .otherwise(0)).drop("Numerad","Denom","Volumen","Bruto")
cat_df = cat_df.withColumn(
    "CajasxLayer", F.col("CajasxLayer")/10).withColumn(
    "CajasxPallet", F.col("CajasxPallet")/10
    )

In [0]:
cat_df.show(3)

+--------+---------+-----------+------------+-------------+-----+------+------+-------+------+-------+-------+---+
|Material|CajasxTSP|CajasxLayer|CajasxPallet|LayersxPallet|VolCS|VolLay|VolPal|VolUnid|PesoCS|PesoLay|PesoPal| Un|
+--------+---------+-----------+------------+-------------+-----+------+------+-------+------+-------+-------+---+
|      18|      0.0|      105.0|       525.0|          5.0| NULL| 384.0|1920.0|   NULL| 21.54| 111.15| 555.75| KG|
|      90|      0.0|       75.0|       375.0|          5.0| NULL| 324.0|1620.0|   NULL|  14.5|  80.65| 403.25| KG|
|      20|      0.0|       64.0|       320.0|          5.0| NULL| 276.0|1380.0|   NULL|  21.5| 71.616| 358.08| KG|
+--------+---------+-----------+------------+-------------+-----+------+------+-------+------+-------+-------+---+
only showing top 3 rows


## Crear tabla MARM

In [0]:
(cat_df.write.mode("overwrite").format("delta").option("header", "true")
 .option("overwriteSchema", "true").option("encoding", "UTF-8")
 .save("/mnt/adls/SupplyChain/PBL/MARM_Prod"))


spark.sql("""DROP TABLE IF EXISTS marm""")
spark.sql("""CREATE TABLE IF NOT EXISTS marm
USING DELTA
OPTIONS
(
PATH 'dbfs:/mnt/adls/SupplyChain/PBL/MARM_Prod'
)""")

DataFrame[]